In [ ]:
MODEL_NAME = "two_head_single_2grams"

In [12]:
VOCABULARY = [
    c for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
]
CONTEXT_LENGTH = 64
EMBEDDING_DIM = 16
NUM_HEADS = 2
NGRAM_SIZE = 2

import copy_transformer.training
import transformer_lens
import infra.dataset_configs as dataset_configs
import copy_transformer.tokenizer
from pathlib import Path

tokenizer = copy_transformer.tokenizer.SingleCharTokenizer(
    alphabet=VOCABULARY,
    bos_token=">",
    eos_token="<",
    unk_token="?",
    pad_token="_",
)

training_dataset_config = dataset_configs.UniqueNgramPatternConfig(
    vocabulary=VOCABULARY,
    n=NGRAM_SIZE,
    separator=tokenizer.bos_token,
    max_pattern_length=CONTEXT_LENGTH // 2,
    min_pattern_length=NGRAM_SIZE * 4,
    use_only_n_unique_tokens_per_pattern=8,
    iterable=False,
    length=100_000,
)

training_config = copy_transformer.training.TrainingConfig(
    model_name=MODEL_NAME,
    epochs=10,
    dataset_config=training_dataset_config,
    validation_dataset_config=training_dataset_config,
)

model_config = transformer_lens.HookedTransformerConfig(
    d_model=EMBEDDING_DIM,
    n_heads=NUM_HEADS,
    d_head=EMBEDDING_DIM // NUM_HEADS,
    n_layers=2,
    n_ctx=CONTEXT_LENGTH,
    attn_only=True,
    d_vocab=tokenizer.vocab_size,
)

model = copy_transformer.training.train_transformer(
    config=training_config,
    model_config=model_config,
    tokenizer=tokenizer,
)

Initializing model from config...
Creating training dataset...
Creating validation dataset...


/home/juliusk/_/Uni/Kurse_Winter_2025_2026/AI Safety Incubator/Research_Project/SubspacePartition/copy_transformer/training.py:224: UserWarning: Some sequences in the training set are longer than 63 (context length - 1 for the BOS token), they will be truncated.
  warnings.warn(


Epoch 1/10, Validation Loss: 3.4164
Epoch 2/10, Validation Loss: 3.1894
Epoch 3/10, Validation Loss: 3.0901
Epoch 4/10, Validation Loss: 3.0750
Epoch 5/10, Validation Loss: 3.0271
Epoch 6/10, Validation Loss: 3.0330
Epoch 7/10, Validation Loss: 2.9950
Epoch 8/10, Validation Loss: 2.9845
Epoch 9/10, Validation Loss: 2.9806
Epoch 10/10, Validation Loss: 2.9468
Model saved to out/models/two_head_single_2-grams_again
  - weights.pt
  - model_config.json
  - tokenizer.json
  - training_args.json


In [13]:
import subspace_partition.visualisation

subspace_partition.visualisation.show_model_attention_patterns(MODEL_NAME)


import subspace_partition.testing

subspace_partition.testing.test_model_performance(MODEL_NAME, test_n_last_tokens=10)

Moving model to device:  cpu


Moving model to device:  cpu


{'num_samples': 1024, 'num_tokens': 10240, 'accuracy': 51.474609375}

In [ ]:
EXPERIMENT_NAME = MODEL_NAME + "_" + ...
ACT_SITES = ["blocks.0.hook_resid_post", "blocks.1.hook_resid_post"]

import subspace_partition.subspace_partition
import infra.dataset_configs
import subspace_partition.model_configs
from pathlib import Path

_, model_training_config = subspace_partition.model_configs.load_model(
    MODEL_NAME, load_training_config=True
)

subspace_partition_dataset_config = model_training_config.dataset_config
subspace_partition_dataset_config.iterable = True
subspace_partition_dataset_config.length = "infinite"

subspace_partition_config = (
    subspace_partition.subspace_partition.SubspacePartitionConfig(
        exp_name=EXPERIMENT_NAME,
        model_name=MODEL_NAME,
        dataset_config=subspace_partition_dataset_config,
        act_sites=ACT_SITES,
        unit_size=2,  # Must divide EMBEDDING_DIM evenly
        max_steps=20_000,
        merge_start=2_000,
        merge_interval=2_000,
        output_dir=Path("out/subspace_partition"),
        search_steps=1,
    )
)

subspace_partition.subspace_partition.run_subspace_partition(
    cfg=subspace_partition_config
)

In [ ]:
import subspace_partition.preimage.cache_act
import subspace_partition.model_configs
import infra.dataset_configs

_, model_training_config = subspace_partition.model_configs.load_model(
    MODEL_NAME, load_training_config=True
)

cached_act_dataset_config = model_training_config.dataset_config
cached_act_dataset_config.length = 10_000

subspace_partition.preimage.cache_act.run_cache_act(
    model_name=MODEL_NAME,
    dataset_config=cached_act_dataset_config,
    act_sites=ACT_SITES,
)

In [ ]:
import subspace_partition.preimage.build_index

subspace_partition.preimage.build_index.run_build_index(
    experiment_name=EXPERIMENT_NAME,
)

In [ ]:
import os
os.environ["INDEX_NAME"] = f"index-{EXPERIMENT_NAME}-cosine"
! ./start_streamlit_app.sh "out/index/$INDEX_NAME"